<a href="https://colab.research.google.com/github/Chosencodes/Lung-Segmentation/blob/main/Lung_Segmentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [28]:
# !wget -q --show-progress https://msd-for-monai.s3-us-west-2.amazonaws.com/Task06_Lung.tar -O /content/Task06_Lung.tar
# !tar -xf /content/Task06_Lung.tar -C /content/
# !ls /content/Task06_Lung        # -> imagesTr  labelsTr  dataset.json

In [29]:
# from google.colab import drive
# drive.mount('/content/drive')

# !mkdir -p "/content/drive/MyDrive/datasets"

# !cp /content/Task06_Lung.tar "/content/drive/MyDrive/datasets/Task06_Lung.tar"
# !ls -lh "/content/drive/MyDrive/datasets/"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!cp "/content/drive/MyDrive/datasets/Task06_Lung.tar" /content/

!tar -xf /content/Task06_Lung.tar -C /content/

data_path = "/content/Task06_Lung"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
#  !ls -la /content/Task06_Lung

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import nibabel as nib
import os

In [ ]:
root = Path("/content/Task06_Lung/imagesTr")
label = Path("/content/Task06_Lung/labelsTr")

In [ ]:
sample_path = next(root.glob("lung*.nii.gz"))
sample_label_path = label / sample_path.name

In [ ]:
data = nib.load(sample_path)
label = nib.load(sample_label_path)

In [ ]:
ct_scan = data.get_fdata()
mask = label.get_fdata().astype(np.uint8)

In [ ]:
print(data.shape)
print(label.shape)

print(nib.aff2axcodes(data.affine))
print(nib.aff2axcodes(label.affine))
print(data.header.get_zooms())
print(np.unique(mask))

In [ ]:
tumor_pixels = np.sum(mask == 1)
background_pixels = np.sum(mask == 0)

print(tumor_pixels)
print(background_pixels)

print("Tumor %:", tumor_pixels / mask.size * 100)

In [ ]:
case = "lung_054.nii.gz"
print("Patient:", case)

image = nib.load(f"/content/Task06_Lung/imagesTr/{case}").get_fdata()
label = nib.load(f"/content/Task06_Lung/labelsTr/{case}").get_fdata().astype(np.uint8)

tumor_slices = np.where(label.sum(axis=(0, 1)) > 0)[0]

print("Tumor slices:", tumor_slices)

z = tumor_slices[np.argmax(label.sum(axis=(0, 1))[tumor_slices])]

plt.figure(figsize=(7,7))

plt.imshow(np.clip(image[:, :, z], -1000, 400).T,
           cmap="gray",
           origin="lower")

plt.imshow(np.ma.masked_where(label[:, :, z] == 0,
                              label[:, :, z]).T,
           cmap="autumn",
           alpha=0.6,
           origin="lower")

plt.title(f"{case} - Slice {z}")
plt.axis("off")
plt.show()

# **Preprocessing**

In [ ]:
!pip install torchio nibabel scikit-learn tqdm -q

In [ ]:
import torchio as tio
from sklearn.model_selection import train_test_split,KFold
from torch.utils.data import DataLoader

In [ ]:
root = Path("/content/Task06_Lung/imagesTr")
label = Path("/content/Task06_Lung/labelsTr")

In [ ]:
image_path = sorted(p for p in root.glob("lung*.nii.gz")  if not p.name.startswith("."))
label_path = sorted(p for p in label.glob("lung*.nii.gz") if not p.name.startswith("."))
print("images:", len(image_path), "labels:", len(label_path))

In [ ]:
def make_subject(image_path,label_path):
  subjects = []
  for img_path,lbl_path in zip(image_path,label_path):
    subject = tio.Subject(
        mri = tio.ScalarImage(img_path),
        mask = tio.LabelMap(lbl_path)
    )
    subjects.append(subject)
  return subjects

all_subject = make_subject(image_path,label_path)

In [ ]:
base_transform = tio.Compose([
    tio.ToCanonical(),
    tio.Resample(1.5),
    tio.Clamp(-1000,400),
    tio.RescaleIntensity(
        out_min_max=(0, 1),
        in_min_max=(-1000, 400),
    ),
])

In [ ]:
train_transform = tio.Compose([
    base_transform,
    tio.RandomFlip(axes=(0,), flip_probability=0.5),
    tio.RandomAffine(scales=(0.9, 1.1), degrees=10),
    tio.RandomNoise(std=(0, 0.03)),
])

val_transform = base_transform

In [ ]:
kf = KFold(n_splits=3, shuffle=True, random_state=42)
PATCH_SIZE = (64, 64, 64)

sampler = tio.LabelSampler(
    patch_size=PATCH_SIZE,
    label_name="mask",
    label_probabilities={0: 0.5, 1: 0.5},
)

def build_fold_loaders(train_subjects, val_subjects):
    train_ds = tio.SubjectsDataset(train_subjects, transform=train_transform)
    train_queue = tio.Queue(
        subjects_dataset=train_ds,
        max_length=96,
        samples_per_volume=14,
        sampler=sampler,
        num_workers=2,
        shuffle_subjects=True,
        shuffle_patches=True,
    )
    train_loader = tio.SubjectsLoader(train_queue, batch_size=4)

    val_ds = tio.SubjectsDataset(val_subjects, transform=val_transform)
    val_loader = tio.SubjectsLoader(val_ds, batch_size=1, num_workers=2, shuffle=False)

    return train_loader, val_loader

print("Preprocessing setup complete.")

# **Model**

In [ ]:
!pip install monai pytorch-lightning -q

In [ ]:
import torch
import pytorch_lightning as pl
from monai.networks.nets import UNet
from monai.losses import DiceCELoss, TverskyLoss
from monai.metrics import DiceMetric
from monai.inferers import sliding_window_inference
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping, LearningRateMonitor
from pytorch_lightning.loggers import TensorBoardLogger

In [ ]:
def build_model():
  return UNet(
      spatial_dims = 3,
      in_channels = 1,
      out_channels = 1,
      channels=(32, 64, 128, 256, 512),
      strides=(2,2,2,2),
      num_res_units=2,
      dropout=0.15,
  )

In [ ]:
torch.set_float32_matmul_precision("high")

In [ ]:
class LungTumorSegmentation(pl.LightningModule):
    def __init__(self, lr=1e-3):
        super().__init__()
        self.save_hyperparameters()
        self.model = build_model()
        # beta > alpha keeps the recall that Tversky gave us (56% vs DiceCE's 0.25%),
        # but 0.6 (not 0.7) reduces the false-positive reward.
        self.loss_fn = TverskyLoss(sigmoid=True, alpha=0.4, beta=0.6,
                                   smooth_nr=1e-5, smooth_dr=1e-5)
        self.metrics = DiceMetric(include_background=True, reduction="mean")
        self.patch_size = (64, 64, 64)

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        mri  = batch["mri"]["data"]
        mask = batch["mask"]["data"].float()
        loss = self.loss_fn(self.model(mri), mask)
        self.log("train_loss", loss, prog_bar=True, batch_size=mri.shape[0])
        return loss

    def validation_step(self, batch, batch_idx):
        mri  = batch["mri"]["data"]
        mask = batch["mask"]["data"].float()
        logits = sliding_window_inference(
            inputs=mri, roi_size=self.patch_size, sw_batch_size=4,
            predictor=self.forward, overlap=0.5, mode="gaussian")
        loss = self.loss_fn(logits, mask)
        pred = (torch.sigmoid(logits) > 0.5).float()
        self.metrics(pred, mask)

        # log recall/precision — Dice alone hid the real problem last time
        tp = (pred * mask).sum()
        self.log("val_loss", loss, prog_bar=True, batch_size=1)
        self.log("val_recall", tp / (mask.sum() + 1e-8), batch_size=1)
        self.log("val_prec",   tp / (pred.sum() + 1e-8), batch_size=1)
        if batch_idx == 0:
            self.log_images(mri.cpu(), logits.cpu(), mask.cpu(), "val")
        return loss

    def on_validation_epoch_end(self):
        dice = self.metrics.aggregate().item()
        self.metrics.reset()
        self.log("val_dice", dice, prog_bar=True)
        print(f"Val dice: {dice:.4f}")

    def log_images(self, mri, logits, mask, name):
        d = mri.shape[-1] // 2
        pred = (torch.sigmoid(logits) > 0.5).float()
        ct = mri[0, 0, :, :, d]
        fig, axes = plt.subplots(1, 2, figsize=(10, 5))
        for ax, m, t in [(axes[0], mask, "Ground Truth"), (axes[1], pred, "Prediction")]:
            ax.imshow(ct, cmap="gray")
            ov = np.ma.masked_where(m[0, 0, :, :, d] == 0, m[0, 0, :, :, d])
            ax.imshow(ov, alpha=0.6, cmap="autumn"); ax.set_title(t); ax.axis("off")
        plt.suptitle(f"{name} — slice {d}"); plt.tight_layout()
        self.logger.experiment.add_figure(name, fig, self.global_step); plt.close(fig)

    def configure_optimizers(self):
        opt = torch.optim.Adam(self.model.parameters(),
                               lr=self.hparams.lr, weight_decay=1e-5)
        sch = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="max", factor=0.5, patience=6)
        return {"optimizer": opt, "lr_scheduler": {"scheduler": sch, "monitor": "val_dice"}}

In [ ]:
import shutil
shutil.rmtree("/content/drive/MyDrive/lung_checkpoints", ignore_errors=True)

In [ ]:
pl.seed_everything(42, workers=True)
fold_scores = []

for fold, (train_idx, val_idx) in enumerate(kf.split(all_subject)):
    fold_dir = f"/content/drive/MyDrive/lung_checkpoints/fold_{fold+1}"
    done = os.path.join(fold_dir, "done.txt")
    os.makedirs(fold_dir, exist_ok=True)
    if os.path.exists(done):
        fold_scores.append(float(open(done).read().split("=")[1]))
        print(f"Fold {fold+1} done ({fold_scores[-1]:.4f}) — skip"); continue

    print(f"\n===== FOLD {fold+1}/{kf.get_n_splits()} =====")
    tr = [all_subject[i] for i in train_idx]; va = [all_subject[i] for i in val_idx]
    train_loader, val_loader = build_fold_loaders(tr, va)

    model = LungTumorSegmentation(lr=1e-3)
    checkpoint = ModelCheckpoint(monitor="val_dice", mode="max", save_top_k=1,
        save_last=True, dirpath=fold_dir, filename="best-{epoch}-{val_dice:.3f}")
    early_stop = EarlyStopping(monitor="val_dice", mode="max", patience=25)
    lr_mon = LearningRateMonitor(logging_interval="epoch")

    # logs to DRIVE so a disconnect doesn't destroy the curve
    loggers = [
        TensorBoardLogger(save_dir="/content/drive/MyDrive/lung_logs", name=f"fold{fold+1}"),
        CSVLogger(save_dir="/content/drive/MyDrive/lung_logs", name=f"fold{fold+1}_csv"),
    ]

    trainer = pl.Trainer(
        accelerator="gpu", devices=1,
        max_epochs=120, min_epochs=40,
        precision="bf16-mixed",              # bf16: no NaN on A100
        gradient_clip_val=1.0,
        check_val_every_n_epoch=2,           # halves wall-clock
        callbacks=[checkpoint, early_stop, lr_mon],
        logger=loggers, log_every_n_steps=5, num_sanity_val_steps=0)

    last = os.path.join(fold_dir, "last.ckpt")
    trainer.fit(model, train_loader, val_loader,
                ckpt_path=last if os.path.exists(last) else None)

    best = checkpoint.best_model_score.item()
    fold_scores.append(best)
    open(done, "w").write(f"best_dice={best:.4f}")
    print(f"Fold {fold+1} best Dice: {best:.4f}")

print(f"\nPer-fold: {[round(s,4) for s in fold_scores]}")
print(f"Mean Dice: {np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f}")

In [ ]:
# import os, numpy as np

# scores = []
# for fold in range(1, 4):
#     marker = f"/content/drive/MyDrive/lung_checkpoints/fold_{fold}/done.txt"
#     if os.path.exists(marker):
#         with open(marker) as f:
#             content = f.read().strip()
#             dice = float(content.split("=")[1])
#             scores.append(dice)
#             print(f"Fold {fold}: {dice:.4f}")
#     else:
#         print(f"Fold {fold}: not finished (no done.txt)")

# if scores:
#     print(f"\nMean Dice: {np.mean(scores):.4f} ± {np.std(scores):.4f}")

In [ ]:
# !pip install tensorboard -q
# from tensorboard.backend.event_processing.event_accumulator import EventAccumulator
# import glob

# path = sorted(glob.glob("./logs/lung_fold1/version_*/events.out.tfevents.*"))[-1]
# ea = EventAccumulator(path); ea.Reload()
# for e in ea.Scalars("val_dice"):
#     print(f"epoch {e.step:3d}  val_dice {e.value:.4f}")

In [ ]:
# # grab one training batch that should contain tumor
# tmp_train, _ = build_fold_loaders(all_subject[:20], all_subject[20:25])
# batch = next(iter(tmp_train))
# x = batch["mri"]["data"].cuda()
# y = batch["mask"]["data"].float().cuda()

# print("batch shape:", tuple(x.shape))
# print("foreground voxels in batch:", int(y.sum().item()), "/", y.numel())
# print("mask unique values:", torch.unique(y).tolist())
# print("image min/max:", float(x.min()), float(x.max()))

# m = build_model().cuda()
# opt = torch.optim.Adam(m.parameters(), lr=1e-3)
# loss_fn = DiceFocalLoss(sigmoid=True, gamma=2.0, lambda_dice=1.0, lambda_focal=1.0)

# for i in range(201):
#     opt.zero_grad()
#     logits = m(x)
#     loss = loss_fn(logits, y)
#     loss.backward(); opt.step()
#     if i % 40 == 0:
#         with torch.no_grad():
#             pred = (torch.sigmoid(logits) > 0.5).float()
#             dice = (2*(pred*y).sum() / (pred.sum()+y.sum()+1e-8)).item()
#         print(f"step {i:3d}  loss {loss.item():.3f}  dice {dice:.3f}")

In [ ]:
# import torch
# from monai.losses import DiceLoss

# tmp_train, _ = build_fold_loaders(all_subject[:20], all_subject[20:25])
# batch = next(iter(tmp_train))
# x = batch["mri"]["data"].cuda(); y = batch["mask"]["data"].float().cuda()

# m = build_model().cuda()
# opt = torch.optim.Adam(m.parameters(), lr=1e-2)     # higher LR
# loss_fn = DiceLoss(sigmoid=True)                    # pure Dice, strong foreground pull

# for i in range(600):
#     opt.zero_grad(); logits = m(x); loss = loss_fn(logits, y)
#     loss.backward(); opt.step()
#     if i % 100 == 0:
#         with torch.no_grad():
#             pred = (torch.sigmoid(logits) > 0.5).float()
#             dice = (2*(pred*y).sum()/(pred.sum()+y.sum()+1e-8)).item()
#         print(f"step {i:3d}  dice {dice:.3f}")

In [ ]:
# import torch, torchio as tio
# from monai.losses import DiceLoss
# from monai.networks.nets import UNet

# P = (64, 64, 64)
# sampler64 = tio.LabelSampler(patch_size=P, label_name="mask",
#                              label_probabilities={0: 0.2, 1: 0.8})
# ds = tio.SubjectsDataset(all_subject[:20], transform=train_transform)
# q  = tio.Queue(ds, max_length=64, samples_per_volume=8, sampler=sampler64,
#                num_workers=2, shuffle_subjects=True, shuffle_patches=True)
# loader = tio.SubjectsLoader(q, batch_size=2)

# batch = next(iter(loader))
# x = batch["mri"]["data"].cuda(); y = batch["mask"]["data"].float().cuda()
# print(f"foreground: {int(y.sum())} / {y.numel()} ({100*y.mean().item():.3f}%)")

# m = UNet(spatial_dims=3, in_channels=1, out_channels=1,
#          channels=(32,64,128,256,512), strides=(2,2,2,2),
#          num_res_units=2, dropout=0.0).cuda()          # dropout OFF for the test
# opt = torch.optim.Adam(m.parameters(), lr=1e-3)
# loss_fn = DiceLoss(sigmoid=True)

# for i in range(600):
#     opt.zero_grad(); logits = m(x); loss = loss_fn(logits, y)
#     loss.backward(); opt.step()
#     if i % 100 == 0:
#         with torch.no_grad():
#             pred = (torch.sigmoid(logits) > 0.5).float()
#             dice = (2*(pred*y).sum()/(pred.sum()+y.sum()+1e-8)).item()
#         print(f"step {i:3d}  dice {dice:.3f}")

In [ ]:
# import torch
# from monai.inferers import sliding_window_inference

# model = LungTumorSegmentation.load_from_checkpoint(
#     "/content/drive/MyDrive/lung_checkpoints/fold_1/last.ckpt").cuda().eval()

# _, val_loader = build_fold_loaders(all_subject[:20], all_subject[20:25])
# b = next(iter(val_loader))
# x = b["mri"]["data"].cuda(); y = b["mask"]["data"].float().cuda()

# with torch.no_grad():
#     logits = sliding_window_inference(x, (64,64,64), 4, model.forward,
#                                       overlap=0.5, mode="gaussian")
#     p = torch.sigmoid(logits)

# for t in [0.5, 0.9, 0.99]:
#     pred = (p > t).float()
#     dice = (2*(pred*y).sum()/(pred.sum()+y.sum()+1e-8)).item()
#     print(f"thr {t}: pred={int(pred.sum()):>8d}  true={int(y.sum()):>6d}  dice={dice:.4f}")